# Phase 5d Wave 5: Polariton Hawking Radiation — Technical Notebook

**Wave:** 5d-W5 (Polariton Paper)  
**Date:** April 2026  
**Paper:** Paper 12 — *Stimulated Hawking Radiation in Polariton BEC*

This notebook derives the reservoir-corrected predictions for analog Hawking radiation
in polariton BEC. Key result: stimulated detection achieves SNR ~10 with 100 probe
photons, vs. ~10^9 repetitions needed for spontaneous correlations.

---

In [1]:
import numpy as np
from src.core.constants import POLARITON_PLATFORMS, POLARITON_MASS, HBAR, K_B
from src.core.formulas import (
    polariton_hawking_temperature,
    stimulated_hawking_gain,
    stimulated_hawking_snr,
    stimulated_hawking_spectrum,
    dispersive_hawking_correction,
)
from src.core.visualizations import fig_stimulated_hawking_spectrum, fig_polariton_regime_map, COLORS

## 1. Reservoir-Corrected Parameters

The polariton speed of sound is corrected from 1.0 to 0.5 µm/ps due to
excitonic reservoir effects (3 independent measurements: Falque 2025,
Estrecho 2021, Amo 2009 vs. 1 theoretical projection: Jacquet 2022).

In [2]:
# Display platform parameters
for name, params in POLARITON_PLATFORMS.items():
    T_H = polariton_hawking_temperature(
        kappa=params['kappa'],
    )
    D = params['xi'] * params['kappa'] / params['c_s']
    kappa_tau = params['kappa'] * params['tau_cav']
    print(f"{name}:")
    print(f"  c_s = {params['c_s']:.2e} m/s")
    print(f"  xi  = {params['xi']:.2e} m")
    print(f"  kappa = {params['kappa']:.2e} s^-1")
    print(f"  T_H = {T_H*1e3:.1f} mK")
    print(f"  D = xi*kappa/c_s = {D:.3f}")
    print(f"  kappa*tau = {kappa_tau:.1f}")
    print()

Paris_long:
  c_s = 5.00e+05 m/s
  xi  = 3.00e-06 m
  kappa = 5.00e+10 s^-1
  T_H = 60.8 mK
  D = xi*kappa/c_s = 0.300
  kappa*tau = 5.0

Paris_ultralong:
  c_s = 5.00e+05 m/s
  xi  = 3.00e-06 m
  kappa = 5.00e+10 s^-1
  T_H = 60.8 mK
  D = xi*kappa/c_s = 0.300
  kappa*tau = 15.0

Paris_standard:
  c_s = 5.00e+05 m/s
  xi  = 3.00e-06 m
  kappa = 5.00e+10 s^-1
  T_H = 60.8 mK
  D = xi*kappa/c_s = 0.300
  kappa*tau = 0.1



## 2. Stimulated Hawking Gain Spectrum

The stimulated gain $G(\omega) = \Gamma(\omega)/(e^{2\pi\omega/\kappa} - 1)$
peaks at low frequencies. For $\omega < 0.3\kappa$, the gain exceeds 10%
of the probe intensity.

In [3]:
# Compute gain at key frequencies for the Paris long platform
kappa = POLARITON_PLATFORMS['Paris_long']['kappa']

for ratio in [0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0]:
    omega = ratio * kappa
    G = stimulated_hawking_gain(omega, kappa)
    print(f"  omega/kappa = {ratio:.2f}:  G = {G:.4f}")

  omega/kappa = 0.05:  G = 2.7092
  omega/kappa = 0.10:  G = 1.1436
  omega/kappa = 0.20:  G = 0.3978
  omega/kappa = 0.30:  G = 0.1790
  omega/kappa = 0.50:  G = 0.0452
  omega/kappa = 1.00:  G = 0.0019
  omega/kappa = 2.00:  G = 0.0000


In [4]:
# viz-ref: fig_stimulated_hawking_spectrum
fig = fig_stimulated_hawking_spectrum()
fig.show()

## 3. Signal-to-Noise Ratio

The stimulated SNR scales as $\sqrt{N_{\text{shots}} \cdot N_{\text{probe}}} \cdot G(\omega)$.
Even a single-shot measurement with 100 probe photons reaches SNR > 10
at $\omega = 0.1\kappa$.

In [5]:
# SNR at omega = 0.1*kappa for various probe counts
omega_probe = 0.1 * kappa

for n_probe in [10, 100, 1000, 10000]:
    snr_1 = stimulated_hawking_snr(omega_probe, kappa, n_probe, n_shots=1)
    snr_100 = stimulated_hawking_snr(omega_probe, kappa, n_probe, n_shots=100)
    print(f"  N_probe = {n_probe:>5d}:  SNR(1 shot) = {snr_1:.1f},  SNR(100 shots) = {snr_100:.1f}")

  N_probe =    10:  SNR(1 shot) = 3.6,  SNR(100 shots) = 36.2
  N_probe =   100:  SNR(1 shot) = 11.4,  SNR(100 shots) = 114.4
  N_probe =  1000:  SNR(1 shot) = 36.2,  SNR(100 shots) = 361.6
  N_probe = 10000:  SNR(1 shot) = 114.4,  SNR(100 shots) = 1143.6


## 4. Dispersive Corrections

The dispersive parameter $D = \xi\kappa/c_s$ controls the leading correction
to $T_H$. For the Paris platform, $D \approx 0.3$ gives ~10% correction.

In [6]:
# Dispersive correction for each platform
for name, params in POLARITON_PLATFORMS.items():
    D = params['xi'] * params['kappa'] / params['c_s']
    correction = dispersive_hawking_correction(D)
    T_H = polariton_hawking_temperature(kappa=params['kappa'])
    T_eff = T_H * (1 + correction)
    print(f"{name}: D = {D:.3f}, correction = {correction:.4f}, T_eff = {T_eff*1e3:.1f} mK")

Paris_long: D = 0.300, correction = 0.9100, T_eff = 116.1 mK
Paris_ultralong: D = 0.300, correction = 0.9100, T_eff = 116.1 mK
Paris_standard: D = 0.300, correction = 0.9100, T_eff = 116.1 mK


## 5. Thermal Noise Competition

The driven-dissipative noise temperature $T_{\text{noise}} = m^* c_s^2 / k_B \approx 1.3$ K
exceeds $T_H$ by over an order of magnitude, but this noise is spectrally featureless
and does not affect the stimulated measurement (classical mean-field level).

In [7]:
# Noise temperature for Paris long platform
params = POLARITON_PLATFORMS['Paris_long']
m_star = POLARITON_MASS
c_s = params['c_s']
T_noise = m_star * c_s**2 / K_B
T_H = polariton_hawking_temperature(kappa=params['kappa'])

print(f"T_noise = {T_noise:.2f} K")
print(f"T_H = {T_H*1e3:.1f} mK")
print(f"T_noise / T_H = {T_noise / T_H:.0f}")
print(f"n_thermal at T=4K: {1/(np.exp(HBAR * params['kappa'] / (K_B * 4)) - 1):.1f}")

T_noise = 1.27 K
T_H = 60.8 mK
T_noise / T_H = 21
n_thermal at T=4K: 10.0


In [8]:
# viz-ref: fig_polariton_regime_map
fig = fig_polariton_regime_map()
fig.show()

## 6. Provenance

All parameters from `POLARITON_PLATFORMS` in `constants.py`.  
All formulas from `formulas.py` with Lean references.  
c_s corrected via Deep Research Reconciliation Protocol (3 measurements vs. 1 projection).  
Lean: `PolaritonTier1.lean` (6 theorems, 0 sorry).  
Formulas: `stimulated_hawking_gain`, `stimulated_hawking_snr`, `stimulated_hawking_spectrum`, `dispersive_hawking_correction`.